# Retrieval playground — dense, BM25, and hybrid retrieval against local SciFact

Manual playground notebook, not part of the package or the automated test
suite. Requires the repository's `.venv` kernel and a locally extracted
SciFact dataset under `datasets/scifact/` (`corpus.jsonl`, `queries.jsonl`,
`qrels/test.tsv`).

Dataset loading, chunking, embedding (with caching), and index construction
(Phase 5) are unchanged below. The comparison itself uses the Phase 6
retrievers (`DenseRetriever`, `BM25Retriever`, `HybridRetriever`) instead of
calling the indexes directly.

The corpus embedding cache used below is intentionally isolated to this
notebook's own folder (`notebooks/embedding_cache/`) — it is not shared with
`datasets/scifact/` and is not a reusable cache framework.

In [1]:
import hashlib
import json
from pathlib import Path
from time import perf_counter

import numpy as np

from rag_eval_lab.chunking.registry import chunking_registry
from rag_eval_lab.core.models import Chunk, Query, RetrievedChunk
from rag_eval_lab.datasets.registry import dataset_registry
from rag_eval_lab.embedding.registry import embedding_registry
from rag_eval_lab.indexing.registry import dense_index_registry, sparse_index_registry
from rag_eval_lab.retrieval.bm25 import BM25Retriever
from rag_eval_lab.retrieval.dense import DenseRetriever
from rag_eval_lab.retrieval.hybrid import HybridRetriever


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, start.parent):
        if (candidate / "datasets").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root (expected 'datasets' and 'src' folders).")


REPO_ROOT = _find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
DATASET_DIR = REPO_ROOT / "datasets" / "scifact"
CACHE_DIR = NOTEBOOK_DIR / "embedding_cache"

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Embedding cache dir (isolated to this notebook):", CACHE_DIR)

c:\Users\QTF4584\Private\rag-eval-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: c:\Users\QTF4584\Private\rag-eval-lab
Dataset dir: c:\Users\QTF4584\Private\rag-eval-lab\datasets\scifact
Embedding cache dir (isolated to this notebook): c:\Users\QTF4584\Private\rag-eval-lab\notebooks\embedding_cache


## Embedding cache helpers (local to this notebook)

Stores the corpus embedding matrix as float32 in a `.npy` file plus a JSON
manifest describing everything it depends on. Reused only when the manifest
inputs match and the loaded matrix passes basic sanity checks (expected row
count, finite values, unit-normalized rows); otherwise it is recomputed and
replaced. This stays a notebook-local helper — not a cache framework,
database, registry, or index persistence layer.

In [2]:
CACHE_FORMAT_VERSION = 1
EMBEDDINGS_FILENAME = "corpus_embeddings.npy"
MANIFEST_FILENAME = "corpus_embeddings_manifest.json"
FORCE_RECOMPUTE = False  # set True to bypass the cache and recompute corpus embeddings

_MANIFEST_INPUT_KEYS = (
    "cache_format_version",
    "model_name",
    "document_prompt",
    "normalize_embeddings",
    "chunking",
    "chunk_fingerprint",
)


def chunk_fingerprint(chunks: list[Chunk]) -> str:
    """SHA256 over an ordered, unambiguous JSON serialization of chunk ids/texts."""
    payload = json.dumps(
        [[chunk.id, chunk.text] for chunk in chunks],
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def build_manifest(chunks: list[Chunk], model_name: str, document_prompt: str | None) -> dict:
    """Describes everything the cached embedding matrix depends on."""
    first_chunk_metadata = chunks[0].metadata
    chunking_config = {
        "strategy": first_chunk_metadata["strategy"],
        "chunk_size": first_chunk_metadata["chunk_size"],
        "overlap": first_chunk_metadata["overlap"],
        "encoding_name": first_chunk_metadata["encoding_name"],
    }
    return {
        "cache_format_version": CACHE_FORMAT_VERSION,
        "model_name": model_name,
        "document_prompt": document_prompt,
        "normalize_embeddings": True,
        "chunking": chunking_config,
        "chunk_fingerprint": chunk_fingerprint(chunks),
    }


def manifest_inputs_match(manifest: dict, expected: dict) -> bool:
    return all(manifest.get(key) == expected.get(key) for key in _MANIFEST_INPUT_KEYS)


def load_cached_embeddings(
    cache_dir: Path,
    expected_manifest: dict,
    expected_row_count: int,
    force_recompute: bool = False,
) -> np.ndarray | None:
    """Returns the cached matrix if it is valid and matches `expected_manifest`, else None."""
    if force_recompute:
        return None

    matrix_path = cache_dir / EMBEDDINGS_FILENAME
    manifest_path = cache_dir / MANIFEST_FILENAME
    if not matrix_path.exists() or not manifest_path.exists():
        return None

    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None

    if not manifest_inputs_match(manifest, expected_manifest):
        return None

    try:
        matrix = np.load(matrix_path, allow_pickle=False)
    except (OSError, ValueError):
        return None

    if matrix.ndim != 2 or matrix.shape[0] != expected_row_count or matrix.shape[1] <= 0:
        return None
    if not np.isfinite(matrix).all():
        return None
    if not np.allclose(np.linalg.norm(matrix, axis=1), 1.0, atol=1e-3):
        return None

    return matrix


def save_cache(cache_dir: Path, matrix: np.ndarray, manifest: dict) -> None:
    cache_dir.mkdir(parents=True, exist_ok=True)
    matrix = np.ascontiguousarray(matrix, dtype=np.float32)
    np.save(cache_dir / EMBEDDINGS_FILENAME, matrix)
    manifest_with_shape = {**manifest, "matrix_shape": list(matrix.shape)}
    (cache_dir / MANIFEST_FILENAME).write_text(json.dumps(manifest_with_shape, indent=2), encoding="utf-8")


def get_or_compute_corpus_embeddings(
    cache_dir: Path,
    chunks: list[Chunk],
    embedder,
    force_recompute: bool = False,
) -> tuple[np.ndarray, bool]:
    """Returns (embedding_matrix, was_cached). Only calls `embedder.embed_documents` on a cache miss."""
    expected_manifest = build_manifest(chunks, embedder.model_name, embedder.document_prompt)

    cached = load_cached_embeddings(
        cache_dir,
        expected_manifest,
        expected_row_count=len(chunks),
        force_recompute=force_recompute,
    )
    if cached is not None:
        return cached, True

    embeddings = embedder.embed_documents([chunk.text for chunk in chunks])
    vectors = np.ascontiguousarray(embeddings, dtype=np.float32)
    save_cache(cache_dir, vectors, expected_manifest)
    return vectors, False

## Before the embedding phase: dataset and chunking

Everything up to this point (and this section) runs on every execution —
only the embedding step below can be skipped on a cache hit.

In [3]:
dataset = dataset_registry.create("beir", dataset_dir=DATASET_DIR, split="test")
documents = dataset.documents()
cases = dataset.evaluation_cases()
documents_by_id = {document.id: document for document in documents}

chunker = chunking_registry.create("recursive", chunk_size=256, overlap=32)
chunks = [chunk for document in documents for chunk in chunker.chunk(document)]

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Evaluation cases:", len(cases))

Documents: 5183
Chunks: 8627
Evaluation cases: 300


## Embedding phase

The only step that can be skipped on a cache hit. Query embeddings are
always computed normally, further down.

In [4]:
print("Loading embedding model...")
embedder = embedding_registry.create(
    "sentence_transformer",
    model_name="BAAI/bge-base-en-v1.5",
    batch_size=32,
    device="cpu",
    query_prompt="Represent this sentence for searching relevant passages: ",
)

started = perf_counter()
vectors, was_cached = get_or_compute_corpus_embeddings(CACHE_DIR, chunks, embedder, FORCE_RECOMPUTE)
elapsed = perf_counter() - started

print(f"Corpus embeddings: {'loaded from cache' if was_cached else 'computed and cached'} ({elapsed:.2f}s)")

embeddings = vectors.tolist()

assert vectors.ndim == 2
assert len(vectors) == len(chunks)
assert np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=0.001)

print("Embedding dimension:", vectors.shape[1])

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 352.74it/s]


Corpus embeddings: loaded from cache (0.29s)
Embedding dimension: 768


## After the embedding phase: indexes and diagnostics

Chunks are already in hand and the corpus embedding matrix (`vectors` /
`embeddings`) is available either from cache or from a fresh computation.
FAISS and BM25 are rebuilt every run regardless of the cache outcome.

In [5]:
dense_index = dense_index_registry.create("faiss")
started = perf_counter()
dense_index.add(chunks, embeddings)
print("FAISS build seconds:", round(perf_counter() - started, 2))

sparse_index = sparse_index_registry.create("bm25")
started = perf_counter()
sparse_index.build(chunks)
print("BM25 build seconds:", round(perf_counter() - started, 2))

FAISS build seconds: 1.21
BM25 build seconds: 2.1


## Build retrievers (Phase 6)

Retrievers consume the already-built embedder and indexes above; they never
load a model or build an index themselves. `HybridRetriever` only fuses
`dense_top_k` dense candidates with `sparse_top_k` sparse candidates via
Reciprocal Rank Fusion — with 50 of each, it can return at most 100 unique
chunks no matter how large `top_k` is requested later, so its output is a
fused candidate ranking, not a full corpus ranking.

In [6]:
DENSE_TOP_K = 50  # hybrid's dense-side candidate count
SPARSE_TOP_K = 50  # hybrid's sparse-side candidate count

dense_retriever = DenseRetriever(embedder=embedder, index=dense_index)
bm25_retriever = BM25Retriever(index=sparse_index)
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=bm25_retriever,
    dense_top_k=DENSE_TOP_K,
    sparse_top_k=SPARSE_TOP_K,
)

print(
    f"Hybrid candidate pool: {DENSE_TOP_K} dense + {SPARSE_TOP_K} sparse = "
    f"at most {DENSE_TOP_K + SPARSE_TOP_K} unique chunks"
)

Hybrid candidate pool: 50 dense + 50 sparse = at most 100 unique chunks


## Optional diagnostics

These functions are not required to compare retrievers below; they are
deeper checks you can opt into per case (see `run_diagnostics` on
`compare_retrievers`):

- `print_cosine_agreement_diagnostics` verifies the FAISS implementation
  against a direct NumPy cosine computation.
- `print_full_ranking_diagnostics` shows where a gold document appears in a
  full dense/BM25 ranking (not meaningful for hybrid — see above).
- `print_truncation_diagnostics` checks whether chunk text exceeds the
  embedding model's input length.

In [12]:
COSINE_TOLERANCE = 1e-3


def print_cosine_agreement_diagnostics(chunks, vectors, dense_index, query_embedding):
    print("=== FAISS vs direct NumPy cosine ===")
    normalized_vectors = vectors.astype(np.float64)
    normalized_vectors = normalized_vectors / np.linalg.norm(normalized_vectors, axis=1, keepdims=True)
    query_vector = np.asarray(query_embedding, dtype=np.float64)
    query_vector = query_vector / np.linalg.norm(query_vector)
    direct_scores = normalized_vectors @ query_vector

    faiss_results = dense_index.search(query_embedding, top_k=5)
    direct_top_positions = np.argsort(-direct_scores)[:5]
    position_by_chunk_id = {chunk.id: position for position, chunk in enumerate(chunks)}

    print("FAISS top 5:")
    for result in faiss_results:
        print(f"  chunk={result.chunk.id} score={result.score:.6f}")

    print("Direct cosine top 5:")
    for position in direct_top_positions:
        print(f"  chunk={chunks[position].id} score={direct_scores[position]:.6f}")

    for result in faiss_results:
        position = position_by_chunk_id[result.chunk.id]
        direct_score = float(direct_scores[position])
        agrees = abs(direct_score - result.score) <= COSINE_TOLERANCE
        print(f"  agreement chunk={result.chunk.id}: faiss={result.score:.6f} direct={direct_score:.6f} match={agrees}")
        assert agrees, f"FAISS score disagrees with direct cosine for chunk {result.chunk.id}"

    max_direct_score = float(direct_scores.max())
    rank1_position = position_by_chunk_id[faiss_results[0].chunk.id]
    rank1_direct_score = float(direct_scores[rank1_position])
    assert abs(rank1_direct_score - max_direct_score) <= COSINE_TOLERANCE, (
        f"FAISS rank 1 score {rank1_direct_score} does not match the maximum direct cosine score {max_direct_score}"
    )
    print(f"FAISS rank 1 matches maximum direct cosine score ({max_direct_score:.6f}), allowing ties/tolerance.")

In [8]:
def print_full_ranking_diagnostics(name, full_results, first_gold_document_id):
    gold_chunk_result = next(
        (result for result in full_results if result.chunk.document_id == first_gold_document_id),
        None,
    )
    if gold_chunk_result is None:
        print(f"{name}: gold document {first_gold_document_id} has no chunk anywhere in the ranking")
    else:
        print(
            f"{name}: first gold-document chunk -> rank={gold_chunk_result.rank} "
            f"score={gold_chunk_result.score:.6f}"
        )
        print(f"{name}: chunk text preview: {gold_chunk_result.chunk.text[:200]}")

    unique_document_order = []
    seen_documents = set()
    for result in full_results:
        if result.chunk.document_id not in seen_documents:
            seen_documents.add(result.chunk.document_id)
            unique_document_order.append(result.chunk.document_id)

    if first_gold_document_id in unique_document_order:
        document_rank = unique_document_order.index(first_gold_document_id) + 1
        print(f"{name}: gold document rank among unique documents = {document_rank}")
    else:
        print(f"{name}: gold document does not appear in any ranked chunk")

In [9]:
def print_truncation_diagnostics(embedder, chunks, first_gold_document_id):
    print("=== Truncation check ===")
    model = embedder._model
    max_seq_length = model.max_seq_length
    tokenizer = model.tokenizer
    document_prompt = embedder.document_prompt or ""

    token_counts = [
        len(tokenizer(document_prompt + chunk.text, truncation=False, add_special_tokens=True)["input_ids"])
        for chunk in chunks
    ]

    max_input_length = max(token_counts)
    num_exceeding = sum(1 for count in token_counts if count > max_seq_length)

    print(f"Model max_seq_length: {max_seq_length}")
    print(f"Maximum input length (tokens): {max_input_length}")
    print(f"Inputs exceeding max_seq_length: {num_exceeding} / {len(chunks)}")

    gold_chunk_positions = [i for i, chunk in enumerate(chunks) if chunk.document_id == first_gold_document_id]
    gold_exceeding = [i for i in gold_chunk_positions if token_counts[i] > max_seq_length]
    print(
        f"Gold document {first_gold_document_id}: {len(gold_chunk_positions)} chunks, "
        f"{len(gold_exceeding)} exceeding max_seq_length"
    )

## Compare dense, BM25, and hybrid retrieval

`compare_retrievers` runs `retrieve(query, top_k)` on all three retrievers
for a single evaluation case (selected by its index into `cases`) and prints
ranks, scores, chunk ids, and parent document ids side by side. Retrieval
time is labeled as such rather than "search time": dense (and therefore
hybrid, which calls the dense retriever internally) retrieval time now
includes query embedding, unlike the raw FAISS search timer used earlier.
The next cell selects an arbitrary list of case indices (not necessarily a
contiguous range, e.g. `[0, 3]` or `[0, 2, 5, 6, 10]`) and runs the
comparison for each.

In [10]:
def compare_retrievers(case_index: int, top_k: int = 5, run_diagnostics: bool = False) -> None:
    case = cases[case_index]
    query = Query(id=case.query_id, text=case.question)

    print(f"\n{'=' * 80}\nCase index {case_index} (query_id={query.id})\n{'=' * 80}")
    print("Question:", case.question)
    print("Gold document judgments:", case.ground_truth.document_relevance)

    hybrid_pool_size = hybrid_retriever.dense_top_k + hybrid_retriever.sparse_top_k
    if top_k > hybrid_pool_size:
        print(
            f"Note: top_k={top_k} exceeds hybrid's candidate pool "
            f"({hybrid_retriever.dense_top_k} dense + {hybrid_retriever.sparse_top_k} sparse = "
            f"{hybrid_pool_size} unique chunks at most); hybrid cannot return more than that "
            "regardless of top_k — this is a fused candidate ranking, not a full corpus ranking."
        )

    print(
        "Note: 'dense' and 'hybrid' retrieval time include query embedding "
        "(DenseRetriever embeds internally); 'bm25' retrieval time does not."
    )

    for name, retriever in (("dense", dense_retriever), ("bm25", bm25_retriever), ("hybrid", hybrid_retriever)):
        started = perf_counter()
        results = retriever.retrieve(query, top_k=top_k)
        elapsed = perf_counter() - started

        assert all(isinstance(result, RetrievedChunk) for result in results)
        assert [result.rank for result in results] == list(range(1, len(results) + 1))

        print(f"\n{name} retrieval time: {elapsed:.4f}s ({len(results)} result(s))")
        for result in results:
            grade = case.ground_truth.document_relevance.get(result.chunk.document_id)
            print(
                f"Rank {result.rank}, score {result.score:.4f}, "
                f"chunk {result.chunk.id}, document {result.chunk.document_id}, gold grade {grade}"
            )

    if run_diagnostics:
        query_embedding = embedder.embed_queries([case.question])[0]

        positive_document_ids = [
            document_id
            for document_id, grade in case.ground_truth.document_relevance.items()
            if grade > 0
        ]
        assert positive_document_ids, "expected at least one positively judged document for diagnostics"
        first_gold_document_id = positive_document_ids[0]

        print_cosine_agreement_diagnostics(chunks, vectors, dense_index, query_embedding)

        print("\n=== Full ranking diagnostics (first positively judged document; dense/BM25 only) ===")
        full_faiss_results = dense_index.search(query_embedding, top_k=len(chunks))
        full_bm25_results = sparse_index.search(case.question, top_k=len(chunks))
        print_full_ranking_diagnostics("FAISS", full_faiss_results, first_gold_document_id)
        print_full_ranking_diagnostics("BM25", full_bm25_results, first_gold_document_id)

        print_truncation_diagnostics(embedder, chunks, first_gold_document_id)

    print(f"\nPASS: case index {case_index} — dense, BM25, and hybrid retrieval all returned results.")

In [13]:
# Any list of case indices works, not just a contiguous range, e.g. [0, 3] or [0, 2, 5, 6, 10].
CASE_INDICES = list(range(0, 10))
TOP_K = 10

for case_index in CASE_INDICES:
    compare_retrievers(case_index, top_k=TOP_K, run_diagnostics=False)

print(f"\nPASS: compared dense, BM25, and hybrid retrieval for {len(CASE_INDICES)} case(s): {CASE_INDICES}")


Case index 0 (query_id=1)
Question: 0-dimensional biomaterials show inductive properties.
Gold document judgments: {'31715818': 1}
Note: 'dense' and 'hybrid' retrieval time include query embedding (DenseRetriever embeds internally); 'bm25' retrieval time does not.

dense retrieval time: 0.1657s (10 result(s))
Rank 1, score 0.5890, chunk 21456232_chunk_1, document 21456232, gold grade None
Rank 2, score 0.5736, chunk 14103509_chunk_0, document 14103509, gold grade None
Rank 3, score 0.5576, chunk 14082855_chunk_0, document 14082855, gold grade None
Rank 4, score 0.5551, chunk 7583104_chunk_1, document 7583104, gold grade None
Rank 5, score 0.5507, chunk 21456232_chunk_0, document 21456232, gold grade None
Rank 6, score 0.5468, chunk 4346436_chunk_0, document 4346436, gold grade None
Rank 7, score 0.5459, chunk 58050905_chunk_1, document 58050905, gold grade None
Rank 8, score 0.5436, chunk 31715818_chunk_0, document 31715818, gold grade 1
Rank 9, score 0.5435, chunk 25950264_chunk_0, d